# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR\^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) package using the `mlcroissant` library for programmatic FAIR data science in Python.

### Dataset Source
The dataset is defined by a Croissant schema and accessible via a public URL (see below).

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and display basic info
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs. We'll list all record sets, their associated fields (columns), and the `@id` for reference.

In [ ]:
# List all Record Sets in the dataset by @id, name, and field ids

record_sets = list(dataset.record_sets())  # returns list of RecordSet objects

print("Found Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"     - {field.id} ({field.name})")
    print()

# For first record set (example), print a first data sample
if record_sets:
    rs_ex = record_sets[0]
    print(f"Example record from '{rs_ex.name}' (record set @id: {rs_ex.id}):")
    for i, rec in enumerate(dataset.records(record_set=rs_ex.id)):
        print(json.dumps(rec, indent=2))
        if i >= 0:  # Only one example
            break

## 3. Data Extraction
Load data from every record set into a DataFrame. Always use the record set and field `@id`s identified above, ensuring reproducibility and clarity.

_Below: Extracts all records from each record set into pandas DataFrames keyed by the record set `@id`._

In [ ]:
# Extract all data from each record set into DataFrames
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Columns in DataFrame for RecordSet '{rs.name}' (@id: '{rs.id}'):")
    print(df.columns.tolist())
    print("First 3 records:")
    display(df.head(3))
    print("\n---\n")

# For subsequent analysis, select a main record set (by @id) for demonstration
main_record_set = record_sets[0].id if record_sets else None  # Choose the first record set if available

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by value, normalizing numeric fields, and grouping by categorical variables. All field/column references use the unique `@id`.

### Example: Filter by a Numeric Field
_Note: Update the field @id and value threshold as per the fields overview above._

In [ ]:
# Choose a numeric field for analysis
# You may wish to list columns for inspection first:
df = dataframes[main_record_set]
print(f"Available columns in main record set (@id: {main_record_set}):\n", df.columns.tolist())

# Example: Suppose the dataset main table contains a mass spectrometry quant measure like 'cr:normalizedAbundance'
# We'll use a hypothetical field '@id' below; replace with one present in your columns, e.g., 'cr:normalizedAbundance', 'cr:coverage', 'cr:peptideCount', etc.

numeric_field_id = None
for col in df.columns:
    if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to convert some fields to numeric if possible
    from pandas.api.types import is_numeric_dtype
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue

if numeric_field_id is not None:
    print(f"Using numeric field with @id: {numeric_field_id}\n")
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    print(f"Filtering records in '{main_record_set}' where '{numeric_field_id}' > {threshold:.2f}\n")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records (head):")
    display(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' field:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by a likely categorical field (e.g. group by Sample, or condition)
    candidate_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"\nGrouping by field with @id: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print("Grouped mean values (head):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No suitable numeric field found in the main record set.")

## 5. Visualization
Visualize the distribution of a chosen numeric field or compare values by groups using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # Boxplot by group if available (from group_field above)
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"'{numeric_field_id}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
We have loaded and explored the [FAIR\^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using `mlcroissant`.
- The data schema and record set/field organization were reviewed by their `@id`s.
- Data from each record set can be extracted using Croissant's `@id` semantics.
- Numeric fields were filtered, normalized, and grouped to support statistical analysis.
- Visualizations allow basic checks on value distributions and inter-group differences.

For further analysis, refer to the dataset documentation or Croissant schema to link computed results to biological meaning and experimental design.